In [26]:
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
num = 2

def read_csv(file, names=[]):
    index_names = ['unit_number', 'time_cycles']
    setting_names = ['setting_1', 'setting_2', 'setting_3']
    sensor_names = ['s_{}'.format(i+1) for i in range(0,21)]
    col_names = index_names + setting_names + sensor_names
    df = pd.read_csv(f'CMAPSSData/{file}' ,sep='\s+',header=None,index_col=False,names=col_names if names == [] else names)
    return df

train = read_csv(f"train_FD00{num}.txt")
test = read_csv(f"test_FD00{num}.txt")

print(len(train.unit_number.unique()), len(test.unit_number.unique()))

<>:10: SyntaxWarning: invalid escape sequence '\s'
<>:10: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_599533/2622426932.py:10: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(f'CMAPSSData/{file}' ,sep='\s+',header=None,index_col=False,names=col_names if names == [] else names)


260 259


# Check if all train has at least 60 data

In [2]:
(train.groupby("unit_number")["time_cycles"].max() >= 60).value_counts()

time_cycles
True    260
Name: count, dtype: int64

In [3]:
(test.groupby("unit_number")["time_cycles"].max() >= 60).value_counts()

time_cycles
True     221
False     38
Name: count, dtype: int64

In [4]:
train.unit_number.max()

np.int64(260)

In [5]:
test.unit_number.max()

np.int64(259)

# Group normal df

In [6]:
normal_train = train[train.time_cycles <= 30].copy()
normal_train["Fault"] = 0
len(normal_train.unit_number.unique())

260

In [7]:
normal_test = test[test.time_cycles <= 30].copy()
normal_test["Fault"] = 0
len(normal_test.unit_number.unique())

259

# Group fault df

In [8]:
max_cycles = train.groupby("unit_number")["time_cycles"].max()
valid_engines = max_cycles[max_cycles >= 60].index
fault_start = max_cycles[valid_engines] - 29
fault_train = train[
    train["unit_number"].isin(valid_engines) &                        # engine must qualify
    (train["time_cycles"] >= train["unit_number"].map(fault_start))    # cycle >= max - 29
].copy()
fault_train["Fault"] = 1
len(fault_train.unit_number.unique())

260

In [9]:
max_cycles = test.groupby("unit_number")["time_cycles"].max()
valid_engines = max_cycles[max_cycles >= 60].index
fault_start = max_cycles[valid_engines] - 29
fault_test = test[
    test["unit_number"].isin(valid_engines) &                        # engine must qualify
    (test["time_cycles"] >= test["unit_number"].map(fault_start))    # cycle >= max - 29
].copy()
fault_test["Fault"] = 1
len(fault_test.unit_number.unique())

221

In [33]:
normal_test

,unit_number,time_cycles,setting_1,setting_2,setting_3,s_1,s_2,s_3,s_4,s_5,...,s_13,s_14,s_15,s_16,s_17,s_18,s_19,s_20,s_21,Fault
0,1,1,9.9987,0.2502,100.0,489.05,605.03,1497.17,1304.99,10.52,...,2388.18,8114.10,8.6476,0.03,369,2319,100.00,28.42,17.1551,0
1,1,2,20.0026,0.7000,100.0,491.19,607.82,1481.20,1246.11,9.35,...,2388.12,8053.06,9.2405,0.02,364,2324,100.00,24.29,14.8039,0
2,1,3,35.0045,0.8400,100.0,449.44,556.00,1359.08,1128.36,5.48,...,2387.75,8053.04,9.3472,0.02,333,2223,100.00,14.98,8.9125,0
3,1,4,42.0066,0.8410,100.0,445.00,550.17,1349.69,1127.89,3.91,...,2387.72,8066.90,9.3961,0.02,332,2212,100.00,10.35,6.4181,0
4,1,5,24.9985,0.6213,60.0,462.54,536.72,1253.18,1050.69,7.05,...,2028.05,7865.66,10.8682,0.02,305,1915,84.93,14.31,8.5740,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33893,259,26,24.9986,0.6200,60.0,462.54,536.98,1256.44,1040.41,7.05,...,2028.12,7871.52,10.8445,0.02,306,1915,84.93,14.38,8.5761,0
33894,259,27,34.9994,0.8400,100.0,449.44,555.67,1359.18,1118.20,5.48,...,2387.96,8067.29,9.2853,0.02,333,2223,100.00,14.92,8.9735,0
33895,259,28,25.0049,0.6208,60.0,462.54,536.85,1249.92,1041.71,7.05,...,2028.21,7868.03,10.8456,0.02,307,1915,84.93,14.40,8.5244,0
33896,259,29,42.0014,0.8418,100.0,445.00,549.51,1349.88,1121.43,3.91,...,2387.91,8078.46,9.3224,0.02,330,2212,100.00,10.68,6.4297,0


In [21]:
fault_test

,unit_number,time_cycles,setting_1,setting_2,setting_3,s_1,s_2,s_3,s_4,s_5,...,s_13,s_14,s_15,s_16,s_17,s_18,s_19,s_20,s_21,Fault
228,1,229,42.0001,0.8400,100.0,445.00,549.65,1362.28,1133.60,3.91,...,2388.30,8096.26,9.3898,0.02,334,2212,100.0,10.60,6.3928,1
229,1,230,41.9982,0.8402,100.0,445.00,549.99,1362.86,1133.93,3.91,...,2388.34,8105.00,9.4254,0.02,332,2212,100.0,10.51,6.3695,1
230,1,231,42.0056,0.8401,100.0,445.00,549.97,1359.21,1142.27,3.91,...,2388.31,8109.07,9.3725,0.02,332,2212,100.0,10.59,6.2394,1
231,1,232,35.0034,0.8400,100.0,449.44,556.59,1375.01,1147.15,5.48,...,2388.47,8091.30,9.3802,0.02,335,2223,100.0,14.69,8.8451,1
232,1,233,10.0056,0.2511,100.0,489.05,605.13,1512.65,1315.00,10.52,...,2388.25,8150.30,8.7090,0.03,371,2319,100.0,28.44,17.0263,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33986,259,119,35.0015,0.8403,100.0,449.44,555.56,1366.01,1129.47,5.48,...,2388.39,8088.36,9.3215,0.02,334,2223,100.0,14.94,8.9065,1
33987,259,120,42.0066,0.8405,100.0,445.00,549.42,1351.13,1123.86,3.91,...,2388.31,8108.48,9.3542,0.02,332,2212,100.0,10.57,6.4075,1
33988,259,121,42.0061,0.8400,100.0,445.00,549.65,1349.14,1118.91,3.91,...,2388.34,8098.77,9.3836,0.02,331,2212,100.0,10.57,6.4805,1
33989,259,122,0.0024,0.0003,100.0,518.67,642.58,1589.61,1408.16,14.62,...,2388.00,8161.85,8.4279,0.03,393,2388,100.0,39.08,23.3589,1


# Concat

In [10]:
train_class = pd.concat([normal_train, fault_train]).reset_index(drop=True)
train_class

,unit_number,time_cycles,setting_1,setting_2,setting_3,s_1,s_2,s_3,s_4,s_5,...,s_13,s_14,s_15,s_16,s_17,s_18,s_19,s_20,s_21,Fault
0,1,1,34.9983,0.8400,100.0,449.44,555.32,1358.61,1137.23,5.48,...,2387.72,8048.56,9.3461,0.02,334,2223,100.00,14.73,8.8071,0
1,1,2,41.9982,0.8408,100.0,445.00,549.90,1353.22,1125.78,3.91,...,2387.66,8072.30,9.3774,0.02,330,2212,100.00,10.41,6.2665,0
2,1,3,24.9988,0.6218,60.0,462.54,537.31,1256.76,1047.45,7.05,...,2028.03,7864.87,10.8941,0.02,309,1915,84.93,14.08,8.6723,0
3,1,4,42.0077,0.8416,100.0,445.00,549.51,1354.03,1126.38,3.91,...,2387.61,8068.66,9.3528,0.02,329,2212,100.00,10.59,6.4701,0
4,1,5,25.0005,0.6203,60.0,462.54,537.07,1257.71,1047.93,7.05,...,2028.00,7861.23,10.8963,0.02,309,1915,84.93,14.13,8.5286,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15595,260,312,20.0037,0.7000,100.0,491.19,608.79,1495.60,1269.51,9.35,...,2389.02,8169.64,9.3035,0.03,369,2324,100.00,24.36,14.5189,1
15596,260,313,10.0022,0.2510,100.0,489.05,605.81,1514.32,1324.12,10.52,...,2388.42,8245.36,8.7586,0.03,374,2319,100.00,28.10,16.9454,1
15597,260,314,25.0041,0.6200,60.0,462.54,537.48,1276.24,1057.92,7.05,...,2030.33,7971.25,11.0657,0.02,310,1915,84.93,14.19,8.5503,1
15598,260,315,25.0033,0.6220,60.0,462.54,537.84,1272.95,1066.30,7.05,...,2030.35,7972.47,11.0537,0.02,311,1915,84.93,14.05,8.3729,1


In [11]:
test_class = pd.concat([normal_test, fault_test]).reset_index(drop=True)
test_class

,unit_number,time_cycles,setting_1,setting_2,setting_3,s_1,s_2,s_3,s_4,s_5,...,s_13,s_14,s_15,s_16,s_17,s_18,s_19,s_20,s_21,Fault
0,1,1,9.9987,0.2502,100.0,489.05,605.03,1497.17,1304.99,10.52,...,2388.18,8114.10,8.6476,0.03,369,2319,100.00,28.42,17.1551,0
1,1,2,20.0026,0.7000,100.0,491.19,607.82,1481.20,1246.11,9.35,...,2388.12,8053.06,9.2405,0.02,364,2324,100.00,24.29,14.8039,0
2,1,3,35.0045,0.8400,100.0,449.44,556.00,1359.08,1128.36,5.48,...,2387.75,8053.04,9.3472,0.02,333,2223,100.00,14.98,8.9125,0
3,1,4,42.0066,0.8410,100.0,445.00,550.17,1349.69,1127.89,3.91,...,2387.72,8066.90,9.3961,0.02,332,2212,100.00,10.35,6.4181,0
4,1,5,24.9985,0.6213,60.0,462.54,536.72,1253.18,1050.69,7.05,...,2028.05,7865.66,10.8682,0.02,305,1915,84.93,14.31,8.5740,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14364,259,119,35.0015,0.8403,100.0,449.44,555.56,1366.01,1129.47,5.48,...,2388.39,8088.36,9.3215,0.02,334,2223,100.00,14.94,8.9065,1
14365,259,120,42.0066,0.8405,100.0,445.00,549.42,1351.13,1123.86,3.91,...,2388.31,8108.48,9.3542,0.02,332,2212,100.00,10.57,6.4075,1
14366,259,121,42.0061,0.8400,100.0,445.00,549.65,1349.14,1118.91,3.91,...,2388.34,8098.77,9.3836,0.02,331,2212,100.00,10.57,6.4805,1
14367,259,122,0.0024,0.0003,100.0,518.67,642.58,1589.61,1408.16,14.62,...,2388.00,8161.85,8.4279,0.03,393,2388,100.00,39.08,23.3589,1


# Save csv

In [12]:
train_class.to_csv(f"csv/train_FD00{num}.csv", index=False)
test_class.to_csv(f"csv/test_FD00{num}.csv", index=False)

In [13]:
df_train = pd.read_csv(f'csv/train_FD00{num}.csv')
df_test = pd.read_csv(f'csv/test_FD00{num}.csv')

# Normalize

In [14]:
def norm(df):
    for column in df.columns[2:-1]:
        df[column] = (df[column] - df[column].mean()) / df[column].std()
    return df.dropna(axis="columns").drop(["unit_number", "time_cycles", "Fault"], axis=1)

df_train_norm = norm(df_train).copy()
df_test_norm = norm(df_test).copy()

In [15]:
df_train_norm

,setting_1,setting_2,setting_3,s_1,s_2,s_3,s_4,s_5,s_6,s_7,...,s_12,s_13,s_14,s_15,s_16,s_17,s_18,s_19,s_20,s_21
0,0.743922,0.863242,0.411694,-0.890231,-0.663307,-0.605927,-0.605828,-0.705601,-0.663726,-0.604227,...,-0.604867,0.408243,-0.253794,0.010003,-0.765383,-0.542189,-0.050873,0.411694,-0.613146,-0.618419
1,1.217114,0.865816,0.411694,-1.058050,-0.808569,-0.656828,-0.701678,-1.138697,-1.084169,-0.988192,...,-0.986784,0.407769,0.022913,0.052003,-0.765383,-0.686289,-0.127187,0.411694,-1.050624,-1.047336
2,0.067959,0.161033,-2.428832,-0.395091,-1.145995,-1.567760,-1.357395,-0.272505,-0.476454,-0.733720,...,-0.741556,-2.431497,-2.394842,2.087202,-0.765383,-1.442815,-2.187680,-2.428832,-0.678970,-0.641177
3,1.217756,0.868391,0.411694,-1.058050,-0.819021,-0.649178,-0.696656,-1.138697,-1.084169,-0.988534,...,-0.984607,0.407374,-0.019514,0.018993,-0.765383,-0.722314,-0.127187,0.411694,-1.032395,-1.012963
4,0.068074,0.156206,-2.428832,-0.395091,-1.152428,-1.558789,-1.353377,-0.272505,-0.474618,-0.738235,...,-0.740903,-2.431734,-2.437269,2.090154,-0.765383,-1.442815,-2.187680,-2.428832,-0.673907,-0.665437
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15595,-0.269709,0.412696,0.411694,0.687791,0.769750,0.687756,0.501515,0.361968,0.375448,0.349084,...,0.345498,0.418506,1.157485,-0.047161,1.306451,0.718689,0.649833,0.411694,0.362063,0.345876
15596,-0.945807,-1.032270,0.411694,0.606906,0.689883,0.864541,0.958667,0.684721,0.713272,0.748988,...,0.760281,0.413769,2.040060,-0.778340,1.306451,0.898814,0.615145,0.411694,0.740805,0.755530
15597,0.068317,0.155241,-2.428832,-0.395091,-1.141439,-1.383798,-1.269748,-0.272505,-0.472782,-0.737825,...,-0.745039,-2.413339,-1.154901,2.317465,-0.765383,-1.406790,-2.187680,-2.428832,-0.667831,-0.661773
15598,0.068263,0.161677,-2.428832,-0.395091,-1.131791,-1.414868,-1.199598,-0.272505,-0.472782,-0.739877,...,-0.740468,-2.413181,-1.140681,2.301363,-0.765383,-1.370765,-2.187680,-2.428832,-0.682008,-0.691723


In [16]:
df_test_norm

,setting_1,setting_2,setting_3,s_1,s_2,s_3,s_4,s_5,s_6,s_7,...,s_12,s_13,s_14,s_15,s_16,s_17,s_18,s_19,s_20,s_21
0,-0.931945,-1.015716,0.412665,0.592310,0.663005,0.723223,0.834898,0.667440,0.694114,0.742599,...,0.743311,0.413478,0.588578,-0.888893,1.448663,0.739405,0.609718,0.412665,0.747746,0.765022
1,-0.259307,0.418876,0.412665,0.672517,0.737238,0.573141,0.343180,0.347567,0.361241,0.335344,...,0.338685,0.413005,-0.141763,-0.092902,-0.690244,0.560163,0.644255,0.412665,0.334419,0.372782
2,0.749386,0.865391,0.412665,-0.892265,-0.641516,-0.574510,-0.640174,-0.710476,-0.668299,-0.617556,...,-0.610824,0.410089,-0.142002,0.050347,-0.690244,-0.551135,-0.053389,0.412665,-0.597317,-0.610052
3,1.220190,0.868581,0.412665,-1.058675,-0.796633,-0.662755,-0.644099,-1.139708,-1.084845,-0.988470,...,-0.988676,0.409853,0.023832,0.115997,-0.690244,-0.586983,-0.129370,0.412665,-1.060683,-1.026181
4,0.076606,0.167870,-2.423102,-0.401280,-1.154491,-1.569730,-1.288811,-0.281244,-0.480945,-0.738013,...,-0.743474,-2.424834,-2.383996,2.092349,-0.690244,-1.554887,-2.180856,-2.423102,-0.664370,-0.666522
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14364,0.749184,0.866348,0.412665,-0.892265,-0.653223,-0.509384,-0.630904,-0.710476,-0.668299,-0.608623,...,-0.610753,0.415133,0.280600,0.015844,-0.690244,-0.515286,-0.053389,0.412665,-0.601320,-0.611053
14365,1.220190,0.866986,0.412665,-1.058675,-0.816587,-0.649222,-0.677754,-1.139708,-1.083026,-0.991177,...,-0.986235,0.414503,0.521335,0.059745,-0.690244,-0.586983,-0.129370,0.412665,-1.038666,-1.027949
14366,1.220157,0.865391,0.412665,-1.058675,-0.810468,-0.667924,-0.719092,-1.139708,-1.083026,-0.991651,...,-0.984656,0.414739,0.405155,0.099216,-0.690244,-0.622831,-0.129370,0.412665,-1.038666,-1.015771
14367,-1.604073,-1.812747,0.412665,1.702462,1.662083,1.591950,1.696491,1.788364,1.807328,1.815822,...,1.824035,0.412060,1.159905,-1.183849,1.448663,1.599765,1.086326,0.412665,1.814589,1.799973


# Match sensor size

In [17]:
if (len(df_train_norm.columns) <= len(df_test_norm.columns)):
    df_test_norm = df_test_norm[df_train_norm.columns].copy()
else: print("we have a problem")

# Convert to numpy

In [18]:
train_npy = df_train_norm.to_numpy()
test_npy = df_test_norm.to_numpy()

# Reshape to something (N, 30, len(sensor))

In [20]:
train_npy = train_npy.reshape(-1, 30, train_npy.shape[-1])
test_npy = test_npy.reshape(-1, 30, train_npy.shape[-1])

ValueError: cannot reshape array of size 344856 into shape (30,24)

In [ ]:
print(train_npy.shape, test_npy.shape)

In [ ]:
np.save(f"npy/train_FD00{num}", train_npy)
np.save(f"npy/test_FD00{num}", test_npy)
